# 🛍️ Customer Segmentation using Clustering
### Data Mining — AIE323 | Alamein University
**Project 1 — Full Pipeline: EDA → Feature Engineering → Clustering → Evaluation**


In [ ]:
# ─── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import f_oneway

import joblib, os

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
print("✅ All libraries imported successfully!")


## Milestone 1 — Data Collection, Exploration & Preprocessing

In [ ]:
# ─── Load Dataset ─────────────────────────────────────────────────────────
# Uses Mall_Customers.csv if present; otherwise generates realistic synthetic data
CSV_PATH = "Mall_Customers.csv"

if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
    print(f"✅ Loaded real dataset: {df_raw.shape}")
else:
    print("⚠️  Mall_Customers.csv not found — generating synthetic data …")
    np.random.seed(RANDOM_STATE)
    n = 300
    gender   = np.random.choice(['Male','Female'], n)
    age      = np.random.randint(18, 71, n)
    income   = np.random.randint(15, 138, n)
    score    = np.random.randint(1,  100, n)
    # Inject realistic cluster patterns
    for i in range(n):
        if income[i] > 80 and age[i] < 40:
            score[i] = np.clip(score[i] + np.random.randint(20, 40), 1, 99)
        if income[i] < 40:
            score[i] = np.clip(score[i] - np.random.randint(0, 20), 1, 99)
    df_raw = pd.DataFrame({
        'CustomerID':     range(1, n+1),
        'Gender':         gender,
        'Age':            age,
        'Annual Income (k$)':  income,
        'Spending Score (1-100)': score
    })
    print(f"✅ Synthetic dataset created: {df_raw.shape}")

df_raw.head()


In [ ]:
# ─── Basic EDA ────────────────────────────────────────────────────────────
print("Shape:", df_raw.shape)
print("\nMissing values:\n", df_raw.isnull().sum())
print("\nDuplicates:", df_raw.duplicated().sum())
df_raw.describe()


In [ ]:
# ─── EDA Visualizations ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Exploratory Data Analysis — Customer Dataset", fontsize=16, y=1.01)

num_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

# Histograms
for i, col in enumerate(num_cols):
    axes[0, i].hist(df_raw[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'Distribution: {col}')
    axes[0, i].set_xlabel(col); axes[0, i].set_ylabel('Count')

# Box plots
for i, col in enumerate(num_cols):
    sns.boxplot(data=df_raw, x='Gender', y=col, ax=axes[1, i], palette='Set2')
    axes[1, i].set_title(f'{col} by Gender')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Distribution plots saved.")


In [ ]:
# ─── Correlation Heatmap ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
corr = df_raw[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Pair Plot ────────────────────────────────────────────────────────────
pair_fig = sns.pairplot(df_raw[num_cols + ['Gender']], hue='Gender',
                        palette='Set1', corner=True, plot_kws={'alpha':0.6})
pair_fig.fig.suptitle("Pair Plot — Feature Relationships", y=1.02)
pair_fig.savefig('pair_plot.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Preprocessing ────────────────────────────────────────────────────────
df = df_raw.copy()

# Encode gender
le = LabelEncoder()
df['Gender_enc'] = le.fit_transform(df['Gender'])          # Male=1, Female=0

# Drop ID (not a feature)
df = df.drop(columns=['CustomerID', 'Gender'])

# Standardize numerical features
scaler = StandardScaler()
feature_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Gender_enc']
df_scaled = pd.DataFrame(scaler.fit_transform(df[feature_cols]),
                         columns=feature_cols)

# Save scaler for Streamlit app
joblib.dump(scaler, 'scaler.pkl')
print("✅ Preprocessing complete. Scaled dataset shape:", df_scaled.shape)
df_scaled.head()


## Milestone 2 — Feature Engineering & Dimensionality Reduction

In [ ]:
# ─── Feature Engineering: RFM-style scores ───────────────────────────────
# For non-transactional data we derive proxy RFM features
df_fe = df_raw.copy()

# Recency proxy: younger customers with high spending = more recent engagement
df_fe['Recency_Score']   = (100 - df_fe['Age']) * 0.5 + df_fe['Spending Score (1-100)'] * 0.5
# Frequency proxy: income level (higher income → more purchase opportunities)
df_fe['Frequency_Score'] = df_fe['Annual Income (k$)']
# Monetary proxy: spending score directly
df_fe['Monetary_Score']  = df_fe['Spending Score (1-100)']

# Normalize RFM scores to 1–5 quintiles
for col in ['Recency_Score','Frequency_Score','Monetary_Score']:
    df_fe[col+'_Q'] = pd.qcut(df_fe[col], q=5, labels=[1,2,3,4,5]).astype(int)

df_fe['RFM_Total'] = (df_fe['Recency_Score_Q'] +
                      df_fe['Frequency_Score_Q'] +
                      df_fe['Monetary_Score_Q'])

# Encode gender
df_fe['Gender_enc'] = le.transform(df_fe['Gender'])

print("✅ Feature engineering complete.")
df_fe[['Recency_Score_Q','Frequency_Score_Q','Monetary_Score_Q','RFM_Total']].describe()


In [ ]:
# ─── RFM Distribution Visualization ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('RFM Score Distributions', fontsize=14)
for ax, col in zip(axes, ['Recency_Score_Q','Frequency_Score_Q','Monetary_Score_Q','RFM_Total']):
    ax.hist(df_fe[col], bins=10, color='coral', edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_',' ')); ax.set_xlabel('Score')
plt.tight_layout()
plt.savefig('rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── PCA — Dimensionality Reduction ─────────────────────────────────────
pca_features = ['Age','Annual Income (k$)','Spending Score (1-100)',
                'Gender_enc','Recency_Score','Frequency_Score','Monetary_Score']

X_pca_input = df_fe[pca_features].copy()
X_pca_scaled = StandardScaler().fit_transform(X_pca_input)

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_pca_scaled)

# Explained variance plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA — Explained Variance per Component')

axes[1].plot(range(1, len(pca_full.explained_variance_ratio_)+1),
             np.cumsum(pca_full.explained_variance_ratio_), marker='o', color='coral')
axes[1].axhline(0.80, color='gray', linestyle='--', label='80% threshold')
axes[1].set_xlabel('Components'); axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('PCA — Cumulative Explained Variance'); axes[1].legend()
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

# Keep 2 components for visualization + the number reaching 80% for modeling
n_80 = np.argmax(np.cumsum(pca_full.explained_variance_ratio_) >= 0.80) + 1
print(f"✅ Components needed for 80% variance: {n_80}")

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_pca_scaled)
print(f"   2D variance explained: {pca_2d.explained_variance_ratio_.sum():.1%}")


## Milestone 3 — Clustering Model Development & Evaluation

In [ ]:
# ─── Prepare final feature matrix ────────────────────────────────────────
X = df_fe[['Age','Annual Income (k$)','Spending Score (1-100)','Gender_enc']].values
X_scaled_final = StandardScaler().fit_transform(X)
print("Feature matrix shape:", X_scaled_final.shape)


In [ ]:
# ─── K-Means: Elbow + Silhouette to find optimal k ───────────────────────
inertias, sil_scores = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled_final)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled_final, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, marker='o', color='steelblue')
axes[0].set_title('Elbow Method — Inertia vs k')
axes[0].set_xlabel('Number of Clusters (k)'); axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, sil_scores, marker='o', color='coral')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('kmeans_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = K_range.start + np.argmax(sil_scores)
print(f"✅ Best k by Silhouette: {best_k}  (score = {max(sil_scores):.3f})")


In [ ]:
# ─── Final K-Means Model ─────────────────────────────────────────────────
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
km_labels = kmeans.fit_predict(X_scaled_final)
df_fe['KMeans_Cluster'] = km_labels
joblib.dump(kmeans, 'kmeans_model.pkl')
print(f"✅ K-Means fitted with k={best_k}. Cluster sizes:")
print(pd.Series(km_labels).value_counts().sort_index())


In [ ]:
# ─── K-Means 2D Visualization (PCA projection) ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                     c=km_labels, cmap='tab10', alpha=0.7, edgecolors='white', s=60)
centers_2d = pca_2d.transform(StandardScaler().fit_transform(
    df_fe[['Age','Annual Income (k$)','Spending Score (1-100)','Gender_enc']].values
)[:1])  # dummy — centers plotted via scatter colors
ax.set_title(f'K-Means Clusters (k={best_k}) — PCA 2D Projection')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.colorbar(scatter, ax=ax, label='Cluster')
plt.tight_layout()
plt.savefig('kmeans_2d.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── DBSCAN ───────────────────────────────────────────────────────────────
# Use k-distance graph to find good epsilon
nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled_final)
distances, _ = nbrs.kneighbors(X_scaled_final)
distances_sorted = np.sort(distances[:, -1])[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(distances_sorted, color='steelblue')
ax.set_title('k-Distance Graph (k=5) — Choose epsilon at the "knee"')
ax.set_xlabel('Points (sorted)'); ax.set_ylabel('5th Nearest-Neighbor Distance')
ax.axhline(y=0.5, color='red', linestyle='--', label='eps ≈ 0.5')
ax.legend()
plt.tight_layout()
plt.savefig('dbscan_kdistance.png', dpi=150, bbox_inches='tight')
plt.show()

dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled_final)
df_fe['DBSCAN_Cluster'] = db_labels
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = (db_labels == -1).sum()
print(f"✅ DBSCAN: {n_clusters_db} clusters, {n_noise} noise points")


In [ ]:
# ─── Hierarchical (Agglomerative) Clustering ─────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
Z = linkage(X_scaled_final, method='ward')
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=10, show_contracted=True)
ax.set_title('Hierarchical Clustering Dendrogram (Ward linkage)')
ax.set_xlabel('Sample index'); ax.set_ylabel('Distance')
ax.axhline(y=6, color='red', linestyle='--', label='Cut-off')
ax.legend()
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

agg = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
agg_labels = agg.fit_predict(X_scaled_final)
df_fe['Agg_Cluster'] = agg_labels
print(f"✅ Agglomerative Clustering done with k={best_k}")


In [ ]:
# ─── Evaluation Metrics Comparison ───────────────────────────────────────
def eval_metrics(X, labels, name):
    mask = labels != -1          # Exclude DBSCAN noise
    Xm, lm = X[mask], labels[mask]
    if len(set(lm)) < 2:
        return {'Algorithm': name, 'Silhouette': 'N/A', 'Davies-Bouldin': 'N/A', 'Calinski-Harabasz': 'N/A'}
    return {
        'Algorithm':          name,
        'Silhouette':         round(silhouette_score(Xm, lm), 4),
        'Davies-Bouldin':     round(davies_bouldin_score(Xm, lm), 4),
        'Calinski-Harabasz':  round(calinski_harabasz_score(Xm, lm), 2)
    }

results = pd.DataFrame([
    eval_metrics(X_scaled_final, km_labels,  'K-Means'),
    eval_metrics(X_scaled_final, db_labels,  'DBSCAN'),
    eval_metrics(X_scaled_final, agg_labels, 'Agglomerative'),
])
print("\n📊 Evaluation Metrics (Higher Silhouette & CH = better | Lower DB = better)\n")
print(results.to_string(index=False))


In [ ]:
# ─── Cluster Profiling (using best model = K-Means) ──────────────────────
profile_cols = ['Age','Annual Income (k$)','Spending Score (1-100)','RFM_Total']
profile = df_fe.groupby('KMeans_Cluster')[profile_cols].mean().round(1)

# Assign persona names based on income + spending patterns
def assign_persona(row):
    inc = row['Annual Income (k$)']
    sco = row['Spending Score (1-100)']
    if inc >= 70 and sco >= 60: return '💎 High-Value Champions'
    if inc >= 70 and sco <  50: return '💰 Cautious High-Earners'
    if inc <  50 and sco >= 60: return '🛒 Budget Enthusiasts'
    if inc <  50 and sco <  50: return '🔻 At-Risk Low-Spenders'
    return '🌟 Mid-Tier Regulars'

profile['Persona'] = profile.apply(assign_persona, axis=1)
print("\n📌 Cluster Profiles:\n")
print(profile.to_string())
profile.to_csv('cluster_profiles.csv')


In [ ]:
# ─── Radar Chart per Cluster ─────────────────────────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

cols_radar = ['Age','Annual Income (k$)','Spending Score (1-100)','RFM_Total']
radar_df   = df_fe.groupby('KMeans_Cluster')[cols_radar].mean()

# Normalize 0-1 for radar
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min())

angles = np.linspace(0, 2*np.pi, len(cols_radar), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = plt.cm.tab10(np.linspace(0, 0.9, len(radar_norm)))

for (idx, row), color in zip(radar_norm.iterrows(), colors):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, color=color, linewidth=2, label=f'Cluster {idx}')
    ax.fill(angles, vals, color=color, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(cols_radar, fontsize=11)
ax.set_title('Cluster Radar Chart', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ All visualizations and models saved!")


## Summary

| Step | Method | Output |
|------|--------|--------|
| Data | Mall Customers / Synthetic | 200–300 customers |
| Preprocessing | StandardScaler, LabelEncoder | Normalized features |
| Feature Eng. | RFM proxy scores | 3 new features |
| Dim. Reduction | PCA (2D) | 2D projection |
| Clustering | K-Means ✅ DBSCAN ✅ Hierarchical ✅ | Segment labels |
| Evaluation | Silhouette / DB / CH | Comparison table |
| Deployment | Streamlit App | `app.py` |

Run `streamlit run app.py` to launch the interactive dashboard.
